### CIFAR10 데이터 전처리, 모델링 클래스 구현

In [ ]:
import numpy as np
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Activation, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.models import Model


class PreprocessData:
    def __init__(self, valid_size, random_state, scaling=False):
        self.valid_size = valid_size
        self.random_state = random_state
        self.scaling = scaling

    def load_datasets(self):
        (train_images, train_labels), (test_images, test_labels) = cifar10.load_data()
        return train_images, train_labels, test_images, test_labels

    def scaled_pixels(self, images, labels):
        if self.scaling:
            images = np.array(images / 255.0, dtype=np.float32)
        else:
            images = np.array(images, dtype=np.float32)
        labels = np.array(labels, dtype=np.float32)
        return images, labels

    def transform_ohe(self, labels):
        ohe_labels = to_categorical(labels)
        return ohe_labels

    def split_train_valid(self, train_images, train_ohe_labels):
        tr_images, val_images, tr_ohe_labels, val_ohe_labels = train_test_split(train_images, train_ohe_labels,
                                                                                test_size=self.valid_size,
                                                                                random_state=self.random_state)
        return tr_images, val_images, tr_ohe_labels, val_ohe_labels

    def preprocess_data(self):
        # load dataset
        train_images, train_labels, test_images, test_labels = self.load_datasets()
        # convert to float32(not scaling)
        train_images, train_labels = self.scaled_pixels(train_images, train_labels)
        test_images, test_labels = self.scaled_pixels(test_images, test_labels)
        # transform labels into One-hot encoding
        train_ohe_labels = self.transform_ohe(train_labels)
        test_ohe_labels = self.transform_ohe(test_labels)
        # split train, valid data
        tr_images, val_images, tr_ohe_labels, val_ohe_labels = self.split_train_valid(train_images, train_ohe_labels)
        # check shape
        print('Train:', tr_images.shape, tr_ohe_labels.shape)
        print('Valid:', val_images.shape, val_ohe_labels.shape)
        print('Test:', test_images.shape, test_ohe_labels.shape)

        return tr_images, tr_ohe_labels, val_images, val_ohe_labels, test_images, test_ohe_labels


class CnnModel:
    input_size = 32

    @classmethod
    def change_input_size(cls, input_size):
        CnnModel.input_size = input_size

    @staticmethod
    def create_model(verbose=True):
        size = CnnModel.input_size
        input_tensor = Input(shape=(size, size, 3))
        # Block1
        x = Conv2D(filters=32, kernel_size=3, padding='same', kernel_initializer='he_normal', activation='relu')(
            input_tensor)
        x = Conv2D(filters=32, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x = MaxPooling2D(pool_size=2)(x)
        # Block2
        x = Conv2D(filters=64, kernel_size=3, padding='same', kernel_initializer='he_normal', activation='relu')(x)
        x = Conv2D(filters=64, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x = MaxPooling2D(pool_size=2)(x)
        # Block3
        x = Conv2D(filters=128, kernel_size=3, padding='valid', kernel_initializer='he_normal', activation='relu')(x)
        x = Conv2D(filters=128, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x = MaxPooling2D(pool_size=2)(x)
        # Block4
        x = Conv2D(filters=256, kernel_size=3, strides=2, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        # Classfier Layer
        x = Flatten()(x)
        x = Dropout(rate=0.4)(x)
        x = Dense(units=256, kernel_initializer='he_normal', activation='relu')(x)
        x = Dropout(rate=0.3)(x)
        x = Dense(units=64, kernel_initializer='he_normal', activation='relu')(x)
        output = Dense(units=10, activation='softmax')(x)

        model = Model(inputs=input_tensor, outputs=output)
        if verbose:
            model.summary()

        return model

### CIFAR10 데이터셋에 ImageDataGenerator 적용해보기

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [ ]:
# 데이터 로드
datasets = PreprocessData(valid_size=0.15, random_state=42, scaling=False)
tr_images, tr_ohe_labels, val_images, val_ohe_labels, test_images, test_ohe_labels = datasets.preprocess_data()

Train: (42500, 32, 32, 3) (42500, 10)
Valid: (7500, 32, 32, 3) (7500, 10)
Test: (10000, 32, 32, 3) (10000, 10)


In [ ]:
# 모델 설계
model = CnnModel.create_model(verbose=True)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 conv2d (Conv2D)             (None, 32, 32, 32)        896       
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 32)        9248      
                                                                 
 batch_normalization (BatchN  (None, 32, 32, 32)       128       
 ormalization)                                                   
                                                                 
 activation (Activation)     (None, 32, 32, 32)        0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 16, 16, 32)       0         
 )                                                           

In [ ]:
# ImageDataGenerator 객체 생성 -> 적용할 증강 기법 설정
tr_gen = ImageDataGenerator(horizontal_flip=True, vertical_flip=True, rescale=1/255.0, rotation_range=0.45, zoom_range=[0.5, 1.5])
val_gen = ImageDataGenerator(rescale=1/255.0)

In [ ]:
# Numpy Array Iterator 객체 생성하여 모델 인풋으로 배치 사이즈만큼 집어넣을 준비

# tr_gen.flow(x_train, y_train, batch_size=64, shuffle=True) : 데이터와 레이블 배열을 가져온다. 
#   - batch_size 만큼 데이터를 증가시킨다. 

flow_tr_gen = tr_gen.flow(x=tr_images, y=tr_ohe_labels, batch_size=64, shuffle=True)
flow_val_gen = val_gen.flow(x=val_images, y=val_ohe_labels, batch_size=65, shuffle=False)

In [ ]:
# Callaback 설정

# Tensorflow, 케라스 콜백함수 ReduceLROnPlateau() : 
#  - 모델의 개선이 없을 경우, Learning Rate를 조절해 모델의 개선을 유도하는 콜백함수이다.

rlr_call = ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.1, patience=4, verbose=1)

# Tensorflow, 케라스 콜백함수 EarlyStopping() :
#  - 모델을 더 이상 학습을 못할 경우(loss, metric등의 개선이 없을 경우), 학습 도중 미리 학습을 종료시키는 콜백함수이다.
    
es_call = EarlyStopping(monitor='val_loss', mode='min', patience=7, verbose=1)

In [ ]:
# 모델 compile
model.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

C:\Users\user\anaconda3\lib\site-packages\keras\optimizers\optimizer_v2\adam.py:110: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


In [ ]:
# 학습
tr_hist = model.fit(flow_tr_gen, epochs=20, validation_data=flow_val_gen, callbacks=[rlr_call, es_call])

Epoch 1/20
665/665 [==============================] - 56s 83ms/step - loss: 1.8640 - accuracy: 0.3120 - val_loss: 1.7482 - val_accuracy: 0.3496 - lr: 0.0010
Epoch 2/20
665/665 [==============================] - 56s 84ms/step - loss: 1.5824 - accuracy: 0.4238 - val_loss: 1.6357 - val_accuracy: 0.4273 - lr: 0.0010
Epoch 3/20
665/665 [==============================] - 56s 84ms/step - loss: 1.4098 - accuracy: 0.4934 - val_loss: 1.2777 - val_accuracy: 0.5436 - lr: 0.0010
Epoch 4/20
665/665 [==============================] - 57s 85ms/step - loss: 1.2788 - accuracy: 0.5456 - val_loss: 1.2125 - val_accuracy: 0.5717 - lr: 0.0010
Epoch 5/20
665/665 [==============================] - 57s 85ms/step - loss: 1.1856 - accuracy: 0.5811 - val_loss: 1.2102 - val_accuracy: 0.5697 - lr: 0.0010
Epoch 6/20
665/665 [==============================] - 57s 85ms/step - loss: 1.1117 - accuracy: 0.6080 - val_loss: 1.5482 - val_accuracy: 0.5115 - lr: 0.0010
Epoch 7/20
665/665 [==============================] - 57s 

In [ ]:
# 평가
test_gen = ImageDataGenerator(rescale=1/255.0)
flow_test_gen = test_gen.flow(x=test_images, y=test_ohe_labels, batch_size=32, shuffle=False)
test_hist = model.evaluate(flow_test_gen)

313/313 [==============================] - 3s 10ms/step - loss: 0.6360 - accuracy: 0.7783
